# scMultiSim — CellSTIC tutorial

**Dataset:** scMultiSim replicate `re1`–`re8`. See `data/scmultisim/README.md` for dataset preparation details.


## Step 1 — Repository path


In [ ]:
from pathlib import Path
import sys

import torch

# Add project root to path
try:
    project_root = Path(__file__).resolve().parent.parent
except NameError:
    _cwd = Path.cwd()
    project_root = _cwd if (_cwd / "model").is_dir() else _cwd.parent
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")

## Step 2 — Imports


In [ ]:
from utils.tools.seed_utils import set_global_seed
from model import CellSTIC
from pipeline.trainer import CellSTICTrainer
from pipeline.evaluator import CellSTICEvaluator
from pipeline.analyzer import SingleLevelAnalysis
from utils.loader import load_scmultisim
from utils.metrics import SpatialVisualizer
from utils.train import ModelUtils, load_config

set_global_seed()  # once before train/eval; pipeline does not set RNG here


## Step 3 — Configuration


In [ ]:
RE_NUM = 1  # replicate id: 1-8
WORK_DIR = str(project_root / "data" / "scmultisim")
RUN_TRAIN = True
RUN_OPTIONAL_SPATIAL_VIS = True

work_dir = Path(WORK_DIR)
output_path = work_dir / f"re{RE_NUM}" / "output"
raw_path = work_dir / f"re{RE_NUM}" / "raw"
model_path = work_dir / f"re{RE_NUM}" / "model"
preprocess_path = work_dir / f"re{RE_NUM}" / "preprocess"
config_path = work_dir / f"re{RE_NUM}" / "config" / "scmultisim_config.yaml"

for _p in [output_path, raw_path, model_path, preprocess_path]:
    _p.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Working replicate: re{RE_NUM}")
print(f"Work directory:   {work_dir}")
print(f"Raw path:         {raw_path}")
print(f"Preprocess path:  {preprocess_path}")
print(f"Model path:       {model_path}")
print(f"Output path:      {output_path}")
print(f"Config path:      {config_path}")
print(f"Device:           {device}")

## Step 4 — Optional visualization

This cell reproduces the cell-type spatial distribution visualization from the original Python runner script.


In [ ]:
if RUN_OPTIONAL_SPATIAL_VIS:
    rna_adata_ct, _, _, _, _, _ = load_scmultisim(
        raw_path,
        preprocess_path,
        use_cache=True,
    )

    if "cell_type" not in rna_adata_ct.obs:
        raise ValueError(
            "`cell_type` not found in rna_adata.obs; cannot visualize cell-type spatial distribution. "
            "For scmultisim, this requires `raw/meta/cell.csv` with column `cell.type.idx`."
        )

    rna_adata_ct = rna_adata_ct.copy()
    rna_adata_ct.obs["cluster"] = rna_adata_ct.obs["cell_type"].astype(str).astype("category")

    save_path_ct = output_path / "cell_type_distribution" / "spatial_domain.svg"
    save_path_ct.parent.mkdir(parents=True, exist_ok=True)
    SpatialVisualizer.generate_spatial_domain_visualization(
        adata_source=rna_adata_ct,
        save_path=str(save_path_ct),
    )
    print(f"Cell-type spatial distribution saved to {save_path_ct}")
else:
    print("Skipped optional cell-type spatial distribution visualization.")

## Step 5 — Configure, train, and evaluate


In [ ]:
config = load_config(config_path)

(
    rna_adata,
    atac_adata,
    true_labels,
    ligand_receptor_map,
    edge_type_map,
    lr_pair_type_constraints,
) = load_scmultisim(
    raw_path,
    preprocess_path,
    use_cache=True,
)

if RUN_TRAIN:
    model = CellSTIC(config.model, device)
    CellSTICTrainer(
        model,
        config,
        model_path=model_path,
        ligand_receptor_map=ligand_receptor_map,
        lr_pair_type_constraints=lr_pair_type_constraints,
        device=device,
    ).train(
        modality_datas=[rna_adata, atac_adata],
        is_train_ccc=True,
        is_train_feature=True,
    )
else:
    model = ModelUtils.load_model(model_path, config, device)

# Evaluate fused features with clustering-based metrics
evaluator = CellSTICEvaluator(
    model,
    config,
    ligand_receptor_map=ligand_receptor_map,
    model_path=model_path,
    output_path=output_path,
    device=device,
    lr_pair_type_constraints=lr_pair_type_constraints,
)
pos_edge_probs_np, edge_type_map, m0 = evaluator.evaluate_ccc_precision(
    modality_datas=[rna_adata, atac_adata],
    ccc_label=true_labels,
    label_edge_type_map=edge_type_map,
    annotation_key="cell_type",
)

analyzer = SingleLevelAnalysis(
    adata=m0,
    pos_edge_probs_np=pos_edge_probs_np,
    edge_type_map=edge_type_map,
    output_path=output_path,
    cell_type_key="cell_type",
    threshold=0.1,
    lr_filter=["101:2", "102:6", "103:10", "104:8", "105:20", "106:30"],
    ccc_ground=true_labels,
)
# Cell-type x LR-pair communication heatmaps
analyzer.run_cell_type_heatmaps(font_size=15)
# Edge strength vs spatial distance plots
analyzer.run_strength_vs_distance(linewidth=0.5, figsize=(3.2, 2.2))
# Simplified LR heatmaps (ground-truth labels passed as pos_edge_probs_np here)
analyzer.run_simple_heatmaps(point_size=50, font_size=18, pos_edge_probs_np=true_labels)
# Spot-level CCC metrics and summary figures
analyzer.run_spot_level_metrics(figsize=(3.2, 2.2))

print("Training/evaluation complete.")